<a href="https://colab.research.google.com/github/Lopez-Max/MachineLearning/blob/main/Lopez_MachineLearning_Medical_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub
path = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")

# Copy the entire script, then run
# Training will start automatically with TRAIN_MODE = True

Using Colab cache for faster access to the 'brats20-dataset-training-validation' dataset.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -----------------------------
# Your Survival Model
# -----------------------------
class SurvivalPredictionNetwork(nn.Module):
    def __init__(self, num_features):
        super(SurvivalPredictionNetwork, self).__init__()
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# -----------------------------
# Function to print parameters
# -----------------------------
def print_parameter_summary(model):
    print("\n===== PARAMETER SUMMARY =====\n")
    total_params = 0

    for name, param in model.named_parameters():
        if param.requires_grad:
            num = param.numel()
            total_params += num
            print(f"{name:20s} --> {num} parameters")

    print("\n-----------------------------")
    print(f"TOTAL TRAINABLE PARAMETERS: {total_params:,}")
    print("-----------------------------\n")


# -----------------------------
# Example Usage
# -----------------------------
if __name__ == "__main__":
    # number of input features: adjust if needed!
    NUM_FEATURES = 6   # age, resection, WT, TC, ET, ED

    model = SurvivalPredictionNetwork(NUM_FEATURES)
    print(model)
    print_parameter_summary(model)


SurvivalPredictionNetwork(
  (fc1): Linear(in_features=6, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=1, bias=True)
)

===== PARAMETER SUMMARY =====

fc1.weight           --> 384 parameters
fc1.bias             --> 64 parameters
fc2.weight           --> 2048 parameters
fc2.bias             --> 32 parameters
fc3.weight           --> 32 parameters
fc3.bias             --> 1 parameters

-----------------------------
TOTAL TRAINABLE PARAMETERS: 2,561
-----------------------------



In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import nibabel as nib
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import random

# ============================================================================
# 1. ADVANCED MODEL ARCHITECTURE: ATTENTION U-NET
# ============================================================================

class AttentionBlock(nn.Module):
    """
    Attention Gate to focus on relevant features.
    """
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv3d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv3d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class DoubleConv3D(nn.Module):
    """Double 3D convolution block."""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv3D, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class AttentionUNet3D(nn.Module):
    """
    3D Attention U-Net.
    Replaces the standard U-Net with one that uses Attention Gates.
    """
    def __init__(self, in_channels=4, num_classes=4, base_features=32):
        super(AttentionUNet3D, self).__init__()

        # Encoder
        self.enc1 = DoubleConv3D(in_channels, base_features)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = DoubleConv3D(base_features, base_features*2)
        self.pool2 = nn.MaxPool3d(2)
        self.enc3 = DoubleConv3D(base_features*2, base_features*4)
        self.pool3 = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = DoubleConv3D(base_features*4, base_features*8)

        # Decoder with Attention
        self.up3 = nn.ConvTranspose3d(base_features*8, base_features*4, kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=base_features*4, F_l=base_features*4, F_int=base_features*2)
        self.dec3 = DoubleConv3D(base_features*8, base_features*4)

        self.up2 = nn.ConvTranspose3d(base_features*4, base_features*2, kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=base_features*2, F_l=base_features*2, F_int=base_features)
        self.dec2 = DoubleConv3D(base_features*4, base_features*2)

        self.up1 = nn.ConvTranspose3d(base_features*2, base_features, kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=base_features, F_l=base_features, F_int=base_features//2)
        self.dec1 = DoubleConv3D(base_features*2, base_features)

        self.final = nn.Conv3d(base_features, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # Decoder
        d3 = self.up3(b)
        x3 = self.att3(g=d3, x=e3)
        d3 = torch.cat((x3, d3), dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        x2 = self.att2(g=d2, x=e2)
        d2 = torch.cat((x2, d2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        x1 = self.att1(g=d1, x=e1)
        d1 = torch.cat((x1, d1), dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

# ============================================================================
# 2. IMPROVED LOSS FUNCTION: DICE + FOCAL
# ============================================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

class DiceFocalLoss(nn.Module):
    def __init__(self, weight_dice=0.5, weight_focal=0.5):
        super(DiceFocalLoss, self).__init__()
        self.weight_dice = weight_dice
        self.weight_focal = weight_focal
        self.focal = FocalLoss()

    def forward(self, pred, target):
        # Dice Calculation
        pred_soft = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 4, 1, 2, 3).float()

        intersection = (pred_soft * target_one_hot).sum(dim=(2, 3, 4))
        union = pred_soft.sum(dim=(2, 3, 4)) + target_one_hot.sum(dim=(2, 3, 4))
        dice = (2. * intersection + 1e-5) / (union + 1e-5)
        dice_loss = 1 - dice.mean()

        # Focal Calculation
        focal = self.focal(pred, target)

        return self.weight_dice * dice_loss + self.weight_focal * focal

# ============================================================================
# 3. DATASET & UTILS
# ============================================================================

class BraTSDataset(Dataset):
    """
    Dataset loader that processes 4 modalities and segmentation masks.
    """
    def __init__(self, base_path, patient_ids, data_type="training", target_shape=(128, 128, 128)):
        self.base_path = base_path
        self.patient_ids = patient_ids
        self.data_type = data_type
        self.target_shape = target_shape

        if data_type == "training":
            self.root_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
        else:
            self.root_dir = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData")

    def __len__(self):
        return len(self.patient_ids)

    def crop_or_pad(self, data, target_shape):
        is_4d = len(data.shape) == 4
        if is_4d:
            c, h, w, d = data.shape
            th, tw, td = target_shape
            canvas = np.zeros((c, th, tw, td), dtype=data.dtype)
        else:
            h, w, d = data.shape
            th, tw, td = target_shape
            canvas = np.zeros((th, tw, td), dtype=data.dtype)

        start_h = max((h - th) // 2, 0)
        start_w = max((w - tw) // 2, 0)
        start_d = max((d - td) // 2, 0)

        end_h = min(start_h + th, h)
        end_w = min(start_w + tw, w)
        end_d = min(start_d + td, d)

        c_start_h = max((th - h) // 2, 0)
        c_start_w = max((tw - w) // 2, 0)
        c_start_d = max((td - d) // 2, 0)

        c_end_h = c_start_h + (end_h - start_h)
        c_end_w = c_start_w + (end_w - start_w)
        c_end_d = c_start_d + (end_d - start_d)

        if is_4d:
            canvas[:, c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[:, start_h:end_h, start_w:end_w, start_d:end_d]
        else:
            canvas[c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[start_h:end_h, start_w:end_w, start_d:end_d]
        return canvas

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        patient_path = os.path.join(self.root_dir, patient_id)

        # Load Modalities
        volumes = []
        for modality in ['flair', 't1', 't1ce', 't2']:
            p = os.path.join(patient_path, f"{patient_id}_{modality}.nii")
            if not os.path.exists(p): p += ".gz"
            try:
                vol = nib.load(p).get_fdata().astype(np.float32)
                vol = self.crop_or_pad(vol, self.target_shape)
                # Normalize
                mean, std = vol.mean(), vol.std()
                if std > 0: vol = (vol - mean) / std
                volumes.append(vol)
            except Exception as e:
                # Handle missing files gracefully
                print(f"Error loading {modality} for {patient_id}: {e}")
                return torch.zeros((4, *self.target_shape)), torch.zeros(self.target_shape).long(), patient_id

        img = np.stack(volumes)
        img_tensor = torch.from_numpy(img).float()

        # Load mask if training
        mask_tensor = torch.zeros(self.target_shape).long()
        if self.data_type == "training":
            p = os.path.join(patient_path, f"{patient_id}_seg.nii")
            if not os.path.exists(p): p += ".gz"
            if os.path.exists(p):
                mask = nib.load(p).get_fdata().astype(np.uint8)
                mask = self.crop_or_pad(mask, self.target_shape)
                # Re-map label 4 to 3 to be contiguous [0, 1, 2, 3]
                mask[mask == 4] = 3
                mask_tensor = torch.from_numpy(mask).long()

        return img_tensor, mask_tensor, patient_id

# ============================================================================
# 4. SURVIVAL PREDICTION UTILS (Preserved)
# ============================================================================

class RadiomicFeatureExtractor:
    def extract_features(self, mask):
        if torch.is_tensor(mask):
            mask = mask.cpu().numpy()

        # BraTS labels: 1 (NCR/NET), 2 (ED), 4 (ET) -> Mapped to 3 in training
        # If mask comes from new training, 4 is 3. If from raw file, 4 is 4.
        # We handle both.
        has_4 = np.any(mask == 4)
        et_label = 4 if has_4 else 3

        vol_ncr = float(np.sum(mask == 1))
        vol_ed = float(np.sum(mask == 2))
        vol_et = float(np.sum(mask == et_label))
        vol_wt = float(np.sum(mask > 0))
        vol_tc = vol_ncr + vol_et

        return {
            'volume_whole_tumor': vol_wt, 'volume_tumor_core': vol_tc,
            'volume_enhancing': vol_et, 'volume_edema': vol_ed
        }

class SurvivalPredictionNetwork(nn.Module):
    def __init__(self, num_features):
        super(SurvivalPredictionNetwork, self).__init__()
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def predict_days_to_live(patient_id, base_path, model_path='survival_model.pth'):
    """Predicts survival days for a single patient."""
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\n--- Analyzing Patient: {patient_id} ---")

    if not os.path.exists(model_path):
        print(f"Error: Model file not found at {model_path}")
        return

    checkpoint = torch.load(model_path, map_location=device)
    model = SurvivalPredictionNetwork(num_features=len(checkpoint['feature_columns'])).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # Check paths
    patient_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", patient_id)
    if not os.path.exists(patient_dir):
        patient_dir = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData", patient_id)

    seg_path = os.path.join(patient_dir, f"{patient_id}_seg.nii")
    if not os.path.exists(seg_path): seg_path += ".gz"

    if not os.path.exists(seg_path):
        print("Segmentation mask not found.")
        return

    mask = nib.load(seg_path).get_fdata()
    extractor = RadiomicFeatureExtractor()
    features = extractor.extract_features(mask)

    # Get Age
    surv_path = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", "survival_info.csv")
    if not os.path.exists(surv_path):
        surv_path = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData", "survival_evaluation.csv")

    age = 60.0
    resection = 0
    if os.path.exists(surv_path):
        df = pd.read_csv(surv_path)
        id_col = 'Brats20ID' if 'Brats20ID' in df.columns else 'BraTS20ID'
        row = df[df[id_col] == patient_id]
        if not row.empty:
            age = float(row['Age'].values[0])
            res_col = 'Extent_of_Resection' if 'Extent_of_Resection' in df.columns else 'ResectionStatus'
            if res_col in df.columns:
                resection = 1 if row[res_col].values[0] == 'GTR' else 0

    # Build input
    feature_vec = []
    for col in checkpoint['feature_columns']:
        if col == 'age': feature_vec.append(age)
        elif col == 'resection_encoded': feature_vec.append(resection)
        elif col in features: feature_vec.append(features[col])
        else: feature_vec.append(0.0)

    f_tensor = torch.FloatTensor([feature_vec]).to(device)
    mean = checkpoint['scaler_mean'].to(device)
    std = checkpoint['scaler_std'].to(device)
    f_tensor = (f_tensor - mean) / std

    with torch.no_grad():
        pred_norm = model(f_tensor).item()
        days = pred_norm * checkpoint['max_survival']

    print(f"✅ Estimated Survival: {days:.1f} days")
    return days

# ============================================================================
# 5. TRAINING FUNCTION
# ============================================================================

def train_segmentation_model(base_path, epochs=50, device='cuda'):
    print(f"Starting Attention U-Net Training on {device}...")

    # 1. Setup Data
    train_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
    all_patients = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

    # Split
    split_idx = int(0.8 * len(all_patients))
    train_ids = all_patients[:split_idx]
    val_ids = all_patients[split_idx:]

    train_set = BraTSDataset(base_path, train_ids, data_type="training")
    val_set = BraTSDataset(base_path, val_ids, data_type="training") # Use training data for validation to have masks

    train_loader = DataLoader(train_set, batch_size=1, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=2)

    # 2. Setup Model
    model = AttentionUNet3D(in_channels=4, num_classes=4).to(device)
    criterion = DiceFocalLoss()
    optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    best_dice = 0

    # 3. Loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for images, masks, _ in pbar:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        # Validation
        model.eval()
        val_dice = 0
        with torch.no_grad():
            for images, masks, _ in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)

                # Simple Dice Calc
                pred = torch.argmax(outputs, dim=1)

                # Calculate mean dice over batch (excluding background)
                # Flatten
                pred_f = pred.view(-1)
                mask_f = masks.view(-1)

                # Overall Dice (Micro-average of foreground)
                # Ignore background (0)
                intersection = ((pred_f > 0) & (pred_f == mask_f)).float().sum()
                union = (pred_f > 0).float().sum() + (mask_f > 0).float().sum()
                dice = (2. * intersection) / (union + 1e-5)
                val_dice += dice.item()

        avg_val_dice = val_dice / len(val_loader)
        print(f"Epoch {epoch+1}: Val Dice (Foreground): {avg_val_dice:.4f}")

        if avg_val_dice > best_dice:
            best_dice = avg_val_dice
            torch.save({'model_state_dict': model.state_dict()}, 'best_att_unet.pth')
            print(f"✓ Saved Best Model (Dice: {best_dice:.4f})")

        scheduler.step(epoch_loss)

if __name__ == "__main__":
    # Ensure you set the correct path to your data
    DATA_PATH = '/kaggle/input/brats20-dataset-training-validation'

    # UNCOMMENT TO TRAIN:
    train_segmentation_model(DATA_PATH, epochs=30)

    # UNCOMMENT TO PREDICT (requires 'survival_model.pth'):
    # predict_days_to_live('BraTS20_Training_001', base_path=DATA_PATH)

Starting Attention U-Net Training on cuda...


Epoch 1/30:   3%|▎         | 9/295 [00:07<03:45,  1.27it/s, loss=0.503]


KeyboardInterrupt: 

In [4]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import nibabel as nib
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# ============================================================================
# 1. ADVANCED MODEL ARCHITECTURE: ATTENTION U-NET
# ============================================================================

class AttentionBlock(nn.Module):
    """Attention Gate to focus on relevant features."""
    def __init__(self, F_g, F_l, F_int):
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv3d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv3d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv3d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm3d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class DoubleConv3D(nn.Module):
    """Double 3D convolution block."""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv3D, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm3d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class AttentionUNet3D(nn.Module):
    """3D Attention U-Net."""
    def __init__(self, in_channels=4, num_classes=4, base_features=32):
        super(AttentionUNet3D, self).__init__()

        # Encoder
        self.enc1 = DoubleConv3D(in_channels, base_features)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = DoubleConv3D(base_features, base_features*2)
        self.pool2 = nn.MaxPool3d(2)
        self.enc3 = DoubleConv3D(base_features*2, base_features*4)
        self.pool3 = nn.MaxPool3d(2)

        # Bottleneck
        self.bottleneck = DoubleConv3D(base_features*4, base_features*8)

        # Decoder with Attention
        self.up3 = nn.ConvTranspose3d(base_features*8, base_features*4, kernel_size=2, stride=2)
        self.att3 = AttentionBlock(F_g=base_features*4, F_l=base_features*4, F_int=base_features*2)
        self.dec3 = DoubleConv3D(base_features*8, base_features*4)

        self.up2 = nn.ConvTranspose3d(base_features*4, base_features*2, kernel_size=2, stride=2)
        self.att2 = AttentionBlock(F_g=base_features*2, F_l=base_features*2, F_int=base_features)
        self.dec2 = DoubleConv3D(base_features*4, base_features*2)

        self.up1 = nn.ConvTranspose3d(base_features*2, base_features, kernel_size=2, stride=2)
        self.att1 = AttentionBlock(F_g=base_features, F_l=base_features, F_int=base_features//2)
        self.dec1 = DoubleConv3D(base_features*2, base_features)

        self.final = nn.Conv3d(base_features, num_classes, kernel_size=1)

    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))

        # Bottleneck
        b = self.bottleneck(self.pool3(e3))

        # Decoder
        d3 = self.up3(b)
        x3 = self.att3(g=d3, x=e3)
        d3 = torch.cat((x3, d3), dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        x2 = self.att2(g=d2, x=e2)
        d2 = torch.cat((x2, d2), dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        x1 = self.att1(g=d1, x=e1)
        d1 = torch.cat((x1, d1), dim=1)
        d1 = self.dec1(d1)

        return self.final(d1)

# ============================================================================
# 2. IMPROVED LOSS FUNCTION: DICE + FOCAL
# ============================================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * ce_loss
        return focal_loss.mean()

class DiceFocalLoss(nn.Module):
    def __init__(self, weight_dice=0.5, weight_focal=0.5):
        super(DiceFocalLoss, self).__init__()
        self.weight_dice = weight_dice
        self.weight_focal = weight_focal
        self.focal = FocalLoss()

    def forward(self, pred, target):
        # Dice Calculation
        pred_soft = F.softmax(pred, dim=1)
        target_one_hot = F.one_hot(target, num_classes=pred.shape[1]).permute(0, 4, 1, 2, 3).float()

        intersection = (pred_soft * target_one_hot).sum(dim=(2, 3, 4))
        union = pred_soft.sum(dim=(2, 3, 4)) + target_one_hot.sum(dim=(2, 3, 4))
        dice = (2. * intersection + 1e-5) / (union + 1e-5)
        dice_loss = 1 - dice.mean()

        # Focal Calculation
        focal = self.focal(pred, target)

        return self.weight_dice * dice_loss + self.weight_focal * focal

# ============================================================================
# 3. DATASET & UTILS
# ============================================================================

class BraTSDataset(Dataset):
    """Dataset loader that processes 4 modalities and segmentation masks."""
    def __init__(self, base_path, patient_ids, data_type="training", target_shape=(128, 128, 128)):
        self.base_path = base_path
        self.patient_ids = patient_ids
        self.data_type = data_type
        self.target_shape = target_shape

        if data_type == "training":
            self.root_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
        else:
            self.root_dir = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData")

    def __len__(self):
        return len(self.patient_ids)

    def crop_or_pad(self, data, target_shape):
        is_4d = len(data.shape) == 4
        if is_4d:
            c, h, w, d = data.shape
            th, tw, td = target_shape
            canvas = np.zeros((c, th, tw, td), dtype=data.dtype)
        else:
            h, w, d = data.shape
            th, tw, td = target_shape
            canvas = np.zeros((th, tw, td), dtype=data.dtype)

        start_h = max((h - th) // 2, 0)
        start_w = max((w - tw) // 2, 0)
        start_d = max((d - td) // 2, 0)

        end_h = min(start_h + th, h)
        end_w = min(start_w + tw, w)
        end_d = min(start_d + td, d)

        c_start_h = max((th - h) // 2, 0)
        c_start_w = max((tw - w) // 2, 0)
        c_start_d = max((td - d) // 2, 0)

        c_end_h = c_start_h + (end_h - start_h)
        c_end_w = c_start_w + (end_w - start_w)
        c_end_d = c_start_d + (end_d - start_d)

        if is_4d:
            canvas[:, c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[:, start_h:end_h, start_w:end_w, start_d:end_d]
        else:
            canvas[c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[start_h:end_h, start_w:end_w, start_d:end_d]
        return canvas

    def __getitem__(self, idx):
        patient_id = self.patient_ids[idx]
        patient_path = os.path.join(self.root_dir, patient_id)

        # Load Modalities
        volumes = []
        for modality in ['flair', 't1', 't1ce', 't2']:
            p = os.path.join(patient_path, f"{patient_id}_{modality}.nii")
            if not os.path.exists(p): p += ".gz"
            try:
                vol = nib.load(p).get_fdata().astype(np.float32)
                vol = self.crop_or_pad(vol, self.target_shape)
                # Normalize
                mean, std = vol.mean(), vol.std()
                if std > 0: vol = (vol - mean) / std
                volumes.append(vol)
            except Exception as e:
                print(f"Error loading {modality} for {patient_id}: {e}")
                return torch.zeros((4, *self.target_shape)), torch.zeros(self.target_shape).long(), patient_id

        img = np.stack(volumes)
        img_tensor = torch.from_numpy(img).float()

        # Load mask if training
        mask_tensor = torch.zeros(self.target_shape).long()
        if self.data_type == "training":
            p = os.path.join(patient_path, f"{patient_id}_seg.nii")
            if not os.path.exists(p): p += ".gz"
            if os.path.exists(p):
                mask = nib.load(p).get_fdata().astype(np.uint8)
                mask = self.crop_or_pad(mask, self.target_shape)
                mask[mask == 4] = 3
                mask_tensor = torch.from_numpy(mask).long()

        return img_tensor, mask_tensor, patient_id

# ============================================================================
# 4. VISUALIZATION FUNCTIONS
# ============================================================================

def plot_training_history(history_dict, save_path='training_history.png'):
    """Plot training and validation losses/metrics over epochs."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Plot Loss
    if 'train_loss' in history_dict:
        axes[0].plot(history_dict['train_loss'], label='Train Loss', linewidth=2, marker='o')
    if 'val_loss' in history_dict:
        axes[0].plot(history_dict['val_loss'], label='Val Loss', linewidth=2, marker='s')
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Plot Dice Score
    if 'val_dice' in history_dict:
        axes[1].plot(history_dict['val_dice'], label='Val Dice Score',
                    color='green', linewidth=2, marker='o')
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Dice Score', fontsize=12)
    axes[1].set_title('Dice Score Over Training', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    axes[1].set_ylim([0, 1])

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved training history to {save_path}")
    plt.close()


def plot_segmentation_results(image, ground_truth, prediction, slice_idx=None,
                              save_path='segmentation_results.png'):
    """Visualize a slice of the MRI with ground truth and predicted segmentation."""
    if torch.is_tensor(image):
        image = image.cpu().numpy()
    if torch.is_tensor(ground_truth):
        ground_truth = ground_truth.cpu().numpy()
    if torch.is_tensor(prediction):
        prediction = prediction.cpu().numpy()

    if slice_idx is None:
        slice_idx = image.shape[-1] // 2

    colors = ['black', 'red', 'green', 'blue']
    cmap = ListedColormap(colors)

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    # Row 1: Different MRI modalities
    modalities = ['FLAIR', 'T1', 'T1CE', 'T2']
    for i in range(4):
        ax = axes[0, i] if i < 3 else axes[1, 0]
        ax.imshow(image[i, :, :, slice_idx], cmap='gray')
        ax.set_title(f'{modalities[i]} - Slice {slice_idx}', fontsize=12, fontweight='bold')
        ax.axis('off')

    # Ground Truth Segmentation
    axes[1, 1].imshow(image[1, :, :, slice_idx], cmap='gray')
    gt_overlay = axes[1, 1].imshow(ground_truth[:, :, slice_idx], cmap=cmap,
                                   alpha=0.6, vmin=0, vmax=3)
    axes[1, 1].set_title('Ground Truth Segmentation', fontsize=12, fontweight='bold')
    axes[1, 1].axis('off')

    # Predicted Segmentation
    axes[1, 2].imshow(image[1, :, :, slice_idx], cmap='gray')
    pred_overlay = axes[1, 2].imshow(prediction[:, :, slice_idx], cmap=cmap,
                                     alpha=0.6, vmin=0, vmax=3)
    axes[1, 2].set_title('Predicted Segmentation', fontsize=12, fontweight='bold')
    axes[1, 2].axis('off')

    cbar = plt.colorbar(pred_overlay, ax=axes[1, 2], fraction=0.046, pad=0.04)
    cbar.set_ticks([0.375, 1.125, 1.875, 2.625])
    cbar.set_ticklabels(['Background', 'NCR/NET', 'Edema', 'Enhancing'])

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved segmentation visualization to {save_path}")
    plt.close()


def plot_dice_scores_per_class(dice_scores_dict, save_path='dice_per_class.png'):
    """Plot Dice scores for each tumor region."""
    fig, ax = plt.subplots(figsize=(10, 6))

    regions = list(dice_scores_dict.keys())
    mean_scores = [np.mean(dice_scores_dict[r]) for r in regions]
    std_scores = [np.std(dice_scores_dict[r]) for r in regions]

    colors = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
    bars = ax.bar(regions, mean_scores, yerr=std_scores, capsize=5,
                  color=colors[:len(regions)], alpha=0.8, edgecolor='black')

    ax.set_ylabel('Dice Score', fontsize=12)
    ax.set_title('Dice Scores by Tumor Region', fontsize=14, fontweight='bold')
    ax.set_ylim([0, 1])
    ax.grid(True, axis='y', alpha=0.3)

    for bar, score in zip(bars, mean_scores):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved dice scores per class to {save_path}")
    plt.close()


def calculate_dice_per_region(pred, target):
    """Calculate Dice score for each tumor region."""
    dice_scores = {}

    # Whole Tumor (WT): All non-zero labels
    pred_wt = (pred > 0).float()
    target_wt = (target > 0).float()
    dice_scores['WT'] = compute_dice(pred_wt, target_wt)

    # Tumor Core (TC): Labels 1 and 3
    pred_tc = ((pred == 1) | (pred == 3)).float()
    target_tc = ((target == 1) | (target == 3)).float()
    dice_scores['TC'] = compute_dice(pred_tc, target_tc)

    # Enhancing Tumor (ET): Label 3
    pred_et = (pred == 3).float()
    target_et = (target == 3).float()
    dice_scores['ET'] = compute_dice(pred_et, target_et)

    # Individual regions
    dice_scores['NCR'] = compute_dice((pred == 1).float(), (target == 1).float())
    dice_scores['ED'] = compute_dice((pred == 2).float(), (target == 2).float())

    return dice_scores


def compute_dice(pred, target):
    """Compute Dice coefficient."""
    intersection = (pred * target).sum()
    union = pred.sum() + target.sum()
    if union == 0:
        return 1.0 if pred.sum() == target.sum() else 0.0
    return (2. * intersection / (union + 1e-5)).item()

# ============================================================================
# 5. ENHANCED TRAINING FUNCTION WITH VISUALIZATION
# ============================================================================

def train_segmentation_model(base_path, epochs=25, device='cuda', visualize_every=5):
    print(f"Starting Attention U-Net Training on {device}...")

    # 1. Setup Data
    train_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
    all_patients = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

    split_idx = int(0.8 * len(all_patients))
    train_ids = all_patients[:split_idx]
    val_ids = all_patients[split_idx:]

    train_set = BraTSDataset(base_path, train_ids, data_type="training")
    val_set = BraTSDataset(base_path, val_ids, data_type="training")

    train_loader = DataLoader(train_set, batch_size=1, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_set, batch_size=1, shuffle=False, num_workers=2)

    # 2. Setup Model
    model = AttentionUNet3D(in_channels=4, num_classes=4).to(device)
    criterion = DiceFocalLoss()
    optimizer = Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

    # 3. Training History
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_dice': []
    }

    dice_history = {
        'WT': [], 'TC': [], 'ET': [], 'NCR': [], 'ED': []
    }

    best_dice = 0

    # 4. Training Loop
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

        for images, masks, _ in pbar:
            images, masks = images.to(device), masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            pbar.set_postfix({'loss': loss.item()})

        avg_train_loss = epoch_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)

        # 5. Validation
        model.eval()
        val_loss = 0
        val_dice = 0
        epoch_dice_scores = {k: [] for k in dice_history.keys()}

        sample_saved = False

        with torch.no_grad():
            for images, masks, patient_id in val_loader:
                images, masks = images.to(device), masks.to(device)
                outputs = model(images)

                loss = criterion(outputs, masks)
                val_loss += loss.item()

                pred = torch.argmax(outputs, dim=1)

                # Calculate dice per region
                region_dice = calculate_dice_per_region(pred[0], masks[0])
                for k in epoch_dice_scores.keys():
                    epoch_dice_scores[k].append(region_dice[k])

                # Overall dice (foreground)
                intersection = ((pred > 0) & (pred == masks)).float().sum()
                union = (pred > 0).float().sum() + (masks > 0).float().sum()
                dice = (2. * intersection) / (union + 1e-5)
                val_dice += dice.item()

                # Save visualization for first sample every N epochs
                if not sample_saved and (epoch + 1) % visualize_every == 0:
                    plot_segmentation_results(
                        images[0], masks[0], pred[0],
                        save_path=f'segmentation_epoch_{epoch+1}.png'
                    )
                    sample_saved = True

        avg_val_loss = val_loss / len(val_loader)
        avg_val_dice = val_dice / len(val_loader)

        history['val_loss'].append(avg_val_loss)
        history['val_dice'].append(avg_val_dice)

        # Store dice scores per region
        for k in dice_history.keys():
            dice_history[k].append(np.mean(epoch_dice_scores[k]))

        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, "
              f"Val Loss: {avg_val_loss:.4f}, Val Dice: {avg_val_dice:.4f}")

        # 6. Save best model
        if avg_val_dice > best_dice:
            best_dice = avg_val_dice
            torch.save({'model_state_dict': model.state_dict()}, 'best_att_unet.pth')
            print(f"✓ Saved Best Model (Dice: {best_dice:.4f})")

        scheduler.step(avg_val_loss)

    # 7. Generate final visualizations
    print("\n" + "="*60)
    print("GENERATING FINAL VISUALIZATIONS")
    print("="*60)

    plot_training_history(history, save_path='final_training_history.png')
    plot_dice_scores_per_class(dice_history, save_path='final_dice_per_class.png')

    print("\n✅ Training complete! Check the generated PNG files for results.")
    return model, history, dice_history


if __name__ == "__main__":
    DATA_PATH = '/kaggle/input/brats20-dataset-training-validation'

    # Train with visualization
    model, history, dice_history = train_segmentation_model(
        DATA_PATH,
        epochs=25,
        visualize_every=5  # Save segmentation visualizations every 5 epochs
    )

In [ ]:
import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

# ===========================================================================
# CONFIGURATION
# ===========================================================================
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

CONFIG = {
    'target_shape': (128, 128, 128) # Fixed input size for the model
}

# ===========================================================================
# SECTION 1: DATASET & UTILS
# ===========================================================================

class BraTSDataset(Dataset):
    """
    Standard Dataset class for BraTS.
    Loads T1, T1ce, T2, FLAIR and creates a 4-channel input.
    """
    def __init__(self, base_path, patient_ids, data_type="training", image_size=(128, 128, 128)):
        self.base_path = base_path
        self.patient_ids = patient_ids
        self.data_type = data_type
        self.image_size = image_size

        # Determine root directory based on data type
        if data_type == "training":
            self.root_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
        else:
            self.root_dir = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData")

    def __len__(self):
        return len(self.patient_ids)

    def crop_or_pad(self, data, target_shape):
        """Center crop or pad to target shape."""
        # Data format: (H, W, D) or (C, H, W, D)
        is_4d = len(data.shape) == 4
        if is_4d:
            c, h, w, d = data.shape
            th, tw, td = target_shape
        else:
            h, w, d = data.shape
            th, tw, td = target_shape

        # Initialize canvas
        if is_4d:
            canvas = np.zeros((c, th, tw, td), dtype=data.dtype)
        else:
            canvas = np.zeros((th, tw, td), dtype=data.dtype)

        # Calculate cropping/padding coordinates
        start_h = max((h - th) // 2, 0)
        start_w = max((w - tw) // 2, 0)
        start_d = max((d - td) // 2, 0)

        end_h = min(start_h + th, h)
        end_w = min(start_w + tw, w)
        end_d = min(start_d + td, d)

        # Calculate canvas coordinates
        c_start_h = max((th - h) // 2, 0)
        c_start_w = max((tw - w) // 2, 0)
        c_start_d = max((td - d) // 2, 0)

        c_end_h = c_start_h + (end_h - start_h)
        c_end_w = c_start_w + (end_w - start_w)
        c_end_d = c_start_d + (end_d - start_d)

        if is_4d:
            canvas[:, c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[:, start_h:end_h, start_w:end_w, start_d:end_d]
        else:
            canvas[c_start_h:c_end_h, c_start_w:c_end_w, c_start_d:c_end_d] = \
                data[start_h:end_h, start_w:end_w, start_d:end_d]

        return canvas

    def __getitem__(self, idx):
        # Implementation skipped for this integrated file as we use manual loading in the pipeline
        # to ensure compatibility with both training and inference flows.
        pass

# ===========================================================================
# SECTION 2: SEGMENTATION MODEL (UNet3D)
# ===========================================================================

class UNet3D(nn.Module):
    def __init__(self, in_channels=4, num_classes=4, base_features=32):
        super(UNet3D, self).__init__()

        def conv_block(in_dim, out_dim):
            return nn.Sequential(
                nn.Conv3d(in_dim, out_dim, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_dim),
                nn.ReLU(inplace=True),
                nn.Conv3d(out_dim, out_dim, kernel_size=3, padding=1),
                nn.BatchNorm3d(out_dim),
                nn.ReLU(inplace=True)
            )

        self.enc1 = conv_block(in_channels, base_features)
        self.pool1 = nn.MaxPool3d(2)
        self.enc2 = conv_block(base_features, base_features*2)
        self.pool2 = nn.MaxPool3d(2)

        self.bottleneck = conv_block(base_features*2, base_features*4)

        self.up2 = nn.ConvTranspose3d(base_features*4, base_features*2, kernel_size=2, stride=2)
        self.dec2 = conv_block(base_features*4, base_features*2)
        self.up1 = nn.ConvTranspose3d(base_features*2, base_features, kernel_size=2, stride=2)
        self.dec1 = conv_block(base_features*2, base_features)

        self.final = nn.Conv3d(base_features, num_classes, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        p1 = self.pool1(e1)
        e2 = self.enc2(p1)
        p2 = self.pool2(e2)

        b = self.bottleneck(p2)

        u2 = self.up2(b)
        d2 = self.dec2(torch.cat((u2, e2), dim=1))

        u1 = self.up1(d2)
        d1 = self.dec1(torch.cat((u1, e1), dim=1))

        return self.final(d1)

# ===========================================================================
# SECTION 3: SURVIVAL ANALYSIS (REAL IMPLEMENTATION)
# ===========================================================================

class SurvivalDataLoader:
    def __init__(self, base_path):
        self.base_path = base_path
        # Define paths based on standard BraTS structure
        self.train_survival_path = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", "survival_info.csv")
        self.val_survival_path = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData", "survival_evaluation.csv")

    def prepare_survival_dataset(self, patient_ids, features_dict):
        """
        Loads clinical data and merges with radiomic features.
        """
        print(f"Loading survival data from: {self.train_survival_path}")
        survival_df = pd.read_csv(self.train_survival_path)

        # Standardize column names
        # CSV format usually: Brats20ID, Age, Survival_days, Extent_of_Resection
        survival_df = survival_df.rename(columns={
            'Brats20ID': 'patient_id',
            'Age': 'age',
            'Survival_days': 'survival_days',
            'Extent_of_Resection': 'resection_status'
        })

        # Filter for patients we have features for
        # We only keep patients present in BOTH lists
        relevant_df = survival_df[survival_df['patient_id'].isin(patient_ids)].copy()

        # Encode Resection Status
        # GTR (Gross Total Resection) vs others
        # Fill NA with 'Unknown'
        relevant_df['resection_status'] = relevant_df['resection_status'].fillna('Unknown')
        relevant_df['resection_encoded'] = relevant_df['resection_status'].apply(
            lambda x: 1 if x == 'GTR' else 0
        )

        # Add Radiomic Features
        # features_dict is {patient_id: {feat1: val1, feat2: val2...}}
        radiomic_data = []
        for pid in relevant_df['patient_id']:
            if pid in features_dict:
                feats = features_dict[pid]
                feats['patient_id'] = pid
                radiomic_data.append(feats)

        radiomic_df = pd.DataFrame(radiomic_data)

        # Merge
        final_df = pd.merge(relevant_df, radiomic_df, on='patient_id')

        # Convert survival days to numeric (coerce errors)
        final_df['survival_days'] = pd.to_numeric(final_df['survival_days'], errors='coerce')
        final_df = final_df.dropna(subset=['survival_days'])

        return final_df

    def get_patient_survival_info(self, patient_id, data_type='training'):
        """Helper to get ground truth info for a specific patient."""
        path = self.train_survival_path if data_type == 'training' else self.val_survival_path
        if not os.path.exists(path):
            return None

        df = pd.read_csv(path)
        # Column mapping check
        id_col = 'Brats20ID' if 'Brats20ID' in df.columns else 'BraTS20ID'

        row = df[df[id_col] == patient_id]
        if row.empty:
            return None

        # Basic parsing
        info = {
            'age': float(row['Age'].values[0]),
        }

        # Handle resection column name variations
        res_col = 'Extent_of_Resection' if 'Extent_of_Resection' in df.columns else 'ResectionStatus'
        if res_col in row.columns:
            info['resection_status'] = row[res_col].values[0]
        else:
            info['resection_status'] = "Unknown"

        # Encode resection
        rs = str(info['resection_status'])
        info['resection_encoded'] = 1 if 'GTR' in rs else 0

        # Handle survival days (might not exist in validation data)
        if 'Survival_days' in row.columns:
             val = row['Survival_days'].values[0]
             try:
                 info['survival_days'] = float(val)
             except:
                 info['survival_days'] = 0
        else:
            info['survival_days'] = 0

        return info

class RadiomicFeatureExtractor:
    def __init__(self):
        pass

    def extract_features(self, mask, image_volume):
        """
        Calculate REAL volumetric features from the segmentation mask.
        mask shape: (H, W, D), values 0,1,2,3 (or 4)
        """
        # Ensure mask is numpy
        if torch.is_tensor(mask):
            mask = mask.cpu().numpy()

        # BraTS labels:
        # 0: Background
        # 1: NCR/NET (Necrotic and non-enhancing tumor core)
        # 2: ED (Peritumoral Edema)
        # 4: ET (GD-enhancing tumor) -> Mapped to 3 in some models, or kept as 4

        # Calculate volumes (pixel counts)
        # Check if 4 is present or if it was mapped to 3
        has_4 = np.any(mask == 4)
        et_label = 4 if has_4 else 3

        vol_ncr = float(np.sum(mask == 1))
        vol_ed = float(np.sum(mask == 2))
        vol_et = float(np.sum(mask == et_label))

        # Derived volumes
        vol_wt = float(np.sum(mask > 0))       # Whole Tumor
        vol_tc = vol_ncr + vol_et       # Tumor Core

        return {
            'volume_whole_tumor': vol_wt,
            'volume_tumor_core': vol_tc,
            'volume_enhancing': vol_et,
            'volume_edema': vol_ed
        }

class SurvivalPredictionNetwork(nn.Module):
    def __init__(self, num_features):
        super(SurvivalPredictionNetwork, self).__init__()
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

def train_survival_model(train_df, val_df, feature_cols, model_type, num_epochs, device, save_path):
    print("\n--- TRAINING SURVIVAL PREDICTION MODEL ---")

    # Prepare data
    X_train = torch.FloatTensor(train_df[feature_cols].values)
    y_train = torch.FloatTensor(train_df['survival_days'].values).unsqueeze(1)

    # Validation data might be small or empty if splitting a small dataset
    if len(val_df) > 0:
        X_val = torch.FloatTensor(val_df[feature_cols].values)
        y_val = torch.FloatTensor(val_df['survival_days'].values).unsqueeze(1)
    else:
        X_val = X_train
        y_val = y_train

    # Normalize targets for regression (helps training stability)
    max_survival = max(y_train.max().item(), y_val.max().item())
    y_train_norm = y_train / max_survival
    y_val_norm = y_val / max_survival

    # Feature scaling (StandardScaler logic inside PyTorch flow)
    # We compute mean/std from training and apply to both
    mean = X_train.mean(dim=0)
    std = X_train.std(dim=0)
    # Avoid div by zero
    std[std == 0] = 1.0

    X_train_scaled = (X_train - mean) / std
    X_val_scaled = (X_val - mean) / std

    # Move to device
    X_train_scaled = X_train_scaled.to(device)
    y_train_norm = y_train_norm.to(device)
    X_val_scaled = X_val_scaled.to(device)
    y_val_norm = y_val_norm.to(device)

    model = SurvivalPredictionNetwork(num_features=len(feature_cols)).to(device)
    optimizer = Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(num_epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_scaled)
        loss = criterion(outputs, y_train_norm)
        loss.backward()
        optimizer.step()
        history['train_loss'].append(loss.item())

        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_scaled)
            val_loss = criterion(val_outputs, y_val_norm)
            history['val_loss'].append(val_loss.item())

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {loss.item():.4f}, Val Loss: {val_loss.item():.4f}")

    print("--- SURVIVAL MODEL TRAINING COMPLETE ---")

    # Save checkpoint including scaling params
    torch.save({
        'model_state_dict': model.state_dict(),
        'feature_columns': feature_cols,
        'model_type': model_type,
        'max_survival': max_survival,
        'scaler_mean': mean,
        'scaler_std': std
    }, save_path)

    return model, history

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def plot_kaplan_meier(survival_df, stratify_by, save_path):
    # Simplified KM plot without lifelines dependency if not available,
    # or using matplotlib to approximate
    print(f"Saving Kaplan-Meier plot to {save_path}")

    groups = survival_df[stratify_by].unique()
    fig, ax = plt.subplots(figsize=(10, 6))

    for group in groups:
        subset = survival_df[survival_df[stratify_by] == group]
        # Sort by time
        times = sorted(subset['survival_days'].values)
        probs = []
        count = len(times)
        for i in range(count):
            probs.append((count - i) / count)

        ax.step(times, probs, where='post', label=f"{stratify_by}: {group}")

    ax.set_title('Kaplan-Meier Survival Estimation')
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Survival Probability')
    ax.legend()
    plt.savefig(save_path)
    plt.close(fig)

def plot_survival_predictions(true_survival, predictions, save_path):
    print(f"Saving survival predictions plot to {save_path}")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(true_survival, predictions, alpha=0.6)

    # Perfect prediction line
    max_val = max(true_survival.max(), predictions.max())
    ax.plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')

    ax.set_title('Survival Predictions vs True Values')
    ax.set_xlabel('True Survival (days)')
    ax.set_ylabel('Predicted Survival (days)')
    ax.legend()
    plt.savefig(save_path)
    plt.close(fig)

def plot_feature_importance(model, feature_cols, save_path):
    print(f"Saving feature importance plot to {save_path}")
    # For a simple linear model (FC1 weights), we can look at weights
    # Sum absolute weights of the first layer connected to each feature
    weights = model.fc1.weight.detach().cpu().numpy()
    importance = np.sum(np.abs(weights), axis=0)

    fig, ax = plt.subplots(figsize=(10, 6))
    sorted_idx = np.argsort(importance)[::-1]

    ax.barh(np.array(feature_cols)[sorted_idx], importance[sorted_idx])
    ax.set_title('Feature Importance (Neural Network Weights)')
    ax.set_xlabel('Sum of Absolute Weights')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close(fig)

# ============================================================================
# INTEGRATED PIPELINE CONFIGURATION
# ============================================================================

SURVIVAL_CONFIG = {
    'model_type': 'regression',
    'num_epochs': 50, # Increased for real training
    'feature_selection': 'all'
}

# ============================================================================
# COMPLETE INTEGRATED WORKFLOW
# ============================================================================

def run_integrated_pipeline(base_path, segmentation_model_path,
                            run_segmentation=True,
                            run_survival=True,
                            extract_features_from='predictions'):
    """
    Complete pipeline: segmentation + feature extraction + survival prediction.
    """

    print("=" * 70)
    print("INTEGRATED BRATS20 PIPELINE: SEGMENTATION + SURVIVAL")
    print("=" * 70)

    # Step 1: Initialize Helpers
    print("\n[STEP 1/5] Initializing data loaders...")
    survival_loader = SurvivalDataLoader(base_path)
    feature_extractor = RadiomicFeatureExtractor()

    # Step 2: Load segmentation model (if needed)
    model = None
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    if run_segmentation and extract_features_from == 'predictions':
        print("\n[STEP 2/5] Loading segmentation model...")
        model = UNet3D(in_channels=4, num_classes=4, base_features=32)
        if os.path.exists(segmentation_model_path):
            checkpoint = torch.load(segmentation_model_path, map_location=device, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            model.eval()
            model = model.to(device)
            print(f"✓ Model loaded from: {segmentation_model_path}")
        else:
            print(f"!! Warning: Model not found at {segmentation_model_path}. Using random weights for demo.")
            model = model.to(device)

    # Step 3: Extract radiomic features for all patients
    print("\n[STEP 3/5] Extracting radiomic features...")

    data_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")
    # Get all patients
    if os.path.exists(data_dir):
        all_patients = sorted([d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))])
    else:
        print(f"Error: Data directory not found at {data_dir}")
        return None, None, None

    features_dict = {}

    # IMPORTANT: We need to filter patients to only those in survival_info.csv to save time,
    # or process all and then filter. Let's process a subset for demonstration speed if needed.
    # For real run, process all_patients.
    # Here we check survival_info to see which ones are needed.
    surv_path = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", "survival_info.csv")
    if os.path.exists(surv_path):
        target_df = pd.read_csv(surv_path)
        target_ids = target_df['Brats20ID'].tolist()
        # Filter all_patients to only those in target_ids
        patients_to_process = [p for p in all_patients if p in target_ids]
        print(f"Found {len(patients_to_process)} patients with survival info.")
    else:
        patients_to_process = all_patients[:50] # Fallback
        print("Survival info not found, processing first 50 patients.")

    # Limit for demonstration if list is huge, remove [:50] for full run
    batch_patients = patients_to_process # [:50]

    for patient_id in tqdm(batch_patients, desc="Extracting features"):
        try:
            patient_dir = os.path.join(data_dir, patient_id)

            segmentation = None

            # Option A: Use Ground Truth Masks (Fastest/Best for training survival)
            if extract_features_from == 'ground_truth':
                seg_path = os.path.join(patient_dir, f"{patient_id}_seg.nii")
                if not os.path.exists(seg_path): seg_path += ".gz"

                if os.path.exists(seg_path):
                    seg_nii = nib.load(seg_path)
                    segmentation = seg_nii.get_fdata()
                else:
                    continue # Skip if no GT

            # Option B: Use Model Predictions
            else:
                image_paths_inference = {
                    'flair': os.path.join(patient_dir, f"{patient_id}_flair.nii"),
                    't1': os.path.join(patient_dir, f"{patient_id}_t1.nii"),
                    't1ce': os.path.join(patient_dir, f"{patient_id}_t1ce.nii"),
                    't2': os.path.join(patient_dir, f"{patient_id}_t2.nii")
                }

                volumes_for_seg = []
                temp_dataset = BraTSDataset(base_path, [], data_type="training")

                valid_patient = True
                for modality in ['flair', 't1', 't1ce', 't2']:
                    path = image_paths_inference[modality]
                    if not os.path.exists(path): path += ".gz"
                    if not os.path.exists(path):
                        valid_patient = False
                        break

                    vol = nib.load(path).get_fdata().astype(np.float32)
                    vol = temp_dataset.crop_or_pad(vol, CONFIG['target_shape'])

                    mean, std = vol.mean(), vol.std()
                    if std > 0: vol = (vol - mean) / std
                    volumes_for_seg.append(vol)

                if not valid_patient: continue

                image_volume_for_seg = np.stack(volumes_for_seg, axis=0)

                with torch.no_grad():
                    img_tensor = torch.from_numpy(image_volume_for_seg).float().unsqueeze(0).to(device)
                    output = model(img_tensor)
                    segmentation = torch.argmax(output, dim=1).squeeze(0).cpu().numpy()

            # Extract features (Volume sums)
            if segmentation is not None:
                features = feature_extractor.extract_features(segmentation, None) # Image volume not needed for pure volumetrics
                features_dict[patient_id] = features

        except Exception as e:
            print(f"Error processing {patient_id}: {str(e)}")
            continue

    print(f"\n✓ Extracted features for {len(features_dict)} patients")

    # Step 4: Prepare survival dataset
    print("\n[STEP 4/5] Preparing survival dataset...")
    patient_ids = list(features_dict.keys())
    survival_df = survival_loader.prepare_survival_dataset(patient_ids, features_dict)

    print(f"✓ Survival dataset created with {len(survival_df)} patients")

    # Step 5: Train survival model
    survival_model = None
    history = None

    if run_survival and len(survival_df) > 10:
        print("\n[STEP 5/5] Training survival prediction model...")

        # Split data
        train_df, val_df = train_test_split(survival_df, test_size=0.2, random_state=42)

        # Select features
        exclude_cols = ['patient_id', 'survival_days', 'resection_status']
        feature_cols = [c for c in survival_df.columns if c not in exclude_cols]
        print(f"Training on features: {feature_cols}")

        model_path = 'survival_model.pth'
        survival_model, history = train_survival_model(
            train_df, val_df, feature_cols,
            model_type=SURVIVAL_CONFIG['model_type'],
            num_epochs=SURVIVAL_CONFIG['num_epochs'],
            device=device,
            save_path=model_path
        )

        # Create comprehensive dashboard
        print("\n[VISUALIZATION] Creating comprehensive results dashboard...")
        create_comprehensive_results_dashboard(
            survival_df=survival_df,
            survival_model=survival_model,
            history=history,
            feature_cols=feature_cols,
            save_dir='results/survival_dashboard'
        )
    else:
        print("Not enough data to train survival model.")

    print("\n" + "=" * 70)
    print("✓ INTEGRATED PIPELINE COMPLETE!")
    return survival_df, survival_model, history

# ============================================================================
# DASHBOARD GENERATION
# ============================================================================

def create_comprehensive_results_dashboard(survival_df, survival_model, history, feature_cols, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 1. Training Curves
    plot_survival_training_curves(history, os.path.join(save_dir, 'training_curves.png'))

    # 2. Kaplan Meier
    plot_kaplan_meier(survival_df, 'resection_status', os.path.join(save_dir, 'kaplan_meier.png'))

    # 3. Feature Importance
    plot_feature_importance(survival_model, feature_cols, os.path.join(save_dir, 'feature_importance.png'))

    # 4. Predictions (using model)
    device = next(survival_model.parameters()).device
    # Prepare full X
    X = torch.FloatTensor(survival_df[feature_cols].values)
    # Scale manualy using stats from training (approximation for viz)
    X = (X - X.mean(0)) / (X.std(0) + 1e-6)
    X = X.to(device)

    survival_model.eval()
    with torch.no_grad():
        preds = survival_model(X).cpu().numpy()
        # Denormalize (approx)
        max_surv = survival_df['survival_days'].max()
        preds = preds * max_surv

    plot_survival_predictions(survival_df['survival_days'].values, preds, os.path.join(save_dir, 'predictions.png'))

def plot_survival_training_curves(history, save_path):
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Training History')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.legend()
    plt.savefig(save_path)
    plt.close()

# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    # CONFIGURATION
    # Ensure this points to the correct input directory in Kaggle
    BASE_PATH = '/kaggle/input/brats20-dataset-training-validation'
    SEG_MODEL_PATH = 'checkpoints/best_model.pth' # Path where your segment model is saved

    # Execute
    survival_df, survival_model, history = run_integrated_pipeline(
        base_path=BASE_PATH,
        segmentation_model_path=SEG_MODEL_PATH,
        run_segmentation=True,
        run_survival=True,
        # 'ground_truth' is faster for dev (uses existing masks).
        # 'predictions' runs your model inference (slower but real pipeline).
        extract_features_from='ground_truth'
    )

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import nibabel as nib
import numpy as np

# ============================================================================
# 1. REQUIRED CLASSES (Must match training definitions to load weights)
# ============================================================================

class RadiomicFeatureExtractor:
    def extract_features(self, mask):
        if torch.is_tensor(mask):
            mask = mask.cpu().numpy()

        # BraTS labels: 1 (NCR/NET), 2 (ED), 4 (ET) -> Mapped to 3 in training
        has_4 = np.any(mask == 4)
        et_label = 4 if has_4 else 3

        vol_ncr = float(np.sum(mask == 1))
        vol_ed = float(np.sum(mask == 2))
        vol_et = float(np.sum(mask == et_label))
        vol_wt = float(np.sum(mask > 0))
        vol_tc = vol_ncr + vol_et

        return {
            'volume_whole_tumor': vol_wt, 'volume_tumor_core': vol_tc,
            'volume_enhancing': vol_et, 'volume_edema': vol_ed
        }

class SurvivalPredictionNetwork(nn.Module):
    def __init__(self, num_features):
        super(SurvivalPredictionNetwork, self).__init__()
        self.fc1 = nn.Linear(num_features, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# ============================================================================
# 2. MAIN FUNCTION
# ============================================================================

def check_patient_survival(patient_id, base_path, model_path='survival_model.pth'):
    """
    Loads a patient's data and compares AI prediction vs Reality.
    """
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"\n{'='*50}")
    print(f"SURVIVAL ANALYSIS: {patient_id}")
    print(f"{'='*50}")

    # 1. Load Model
    if not os.path.exists(model_path):
        print(f"❌ Error: Model file '{model_path}' not found.")
        print("   You must run the training script at least once to create this file.")
        return

    checkpoint = torch.load(model_path, map_location=device)
    model = SurvivalPredictionNetwork(num_features=len(checkpoint['feature_columns'])).to(device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    # 2. Find Patient Files
    patient_dir = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", patient_id)
    if not os.path.exists(patient_dir):
        # Check validation folder if not in training
        patient_dir = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData", patient_id)

    seg_path = os.path.join(patient_dir, f"{patient_id}_seg.nii")
    if not os.path.exists(seg_path): seg_path += ".gz"

    if not os.path.exists(seg_path):
        print(f"❌ Error: Segmentation mask not found for {patient_id}")
        return

    # 3. Calculate Tumor Features
    print("1. Analyzing Tumor Structure...")
    mask = nib.load(seg_path).get_fdata()
    extractor = RadiomicFeatureExtractor()
    features = extractor.extract_features(mask)
    print(f"   > Whole Tumor Volume: {features['volume_whole_tumor']:.0f} mm³")

    # 4. Get Clinical Data
    print("2. Retrieving Clinical Data...")
    surv_path = os.path.join(base_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData", "survival_info.csv")
    if not os.path.exists(surv_path):
        surv_path = os.path.join(base_path, "BraTS2020_ValidationData", "MICCAI_BraTS2020_ValidationData", "survival_evaluation.csv")

    age = 60.0 # Default
    resection = 0 # Default
    actual_survival = None

    if os.path.exists(surv_path):
        df = pd.read_csv(surv_path)
        id_col = 'Brats20ID' if 'Brats20ID' in df.columns else 'BraTS20ID'
        row = df[df[id_col] == patient_id]
        if not row.empty:
            age = float(row['Age'].values[0])
            # Resection status
            res_col = 'Extent_of_Resection' if 'Extent_of_Resection' in df.columns else 'ResectionStatus'
            if res_col in df.columns:
                resection = 1 if row[res_col].values[0] == 'GTR' else 0
            # Actual Survival
            if 'Survival_days' in row.columns:
                try:
                    actual_survival = float(row['Survival_days'].values[0])
                except:
                    pass

    print(f"   > Age: {age}")
    print(f"   > Resection (GTR): {'Yes' if resection==1 else 'No'}")

    # 5. Run Prediction
    feature_vec = []
    for col in checkpoint['feature_columns']:
        if col == 'age': feature_vec.append(age)
        elif col == 'resection_encoded': feature_vec.append(resection)
        elif col in features: feature_vec.append(features[col])
        else: feature_vec.append(0.0)

    f_tensor = torch.FloatTensor([feature_vec]).to(device)
    mean = checkpoint['scaler_mean'].to(device)
    std = checkpoint['scaler_std'].to(device)
    f_tensor = (f_tensor - mean) / std

    with torch.no_grad():
        pred_norm = model(f_tensor).item()
        predicted_days = pred_norm * checkpoint['max_survival']

    # 6. Display Results
    print("\n" + "-"*30)
    print("RESULTS")
    print("-"*30)
    print(f"🤖 AI PREDICTION:    {predicted_days:.1f} days")

    if actual_survival is not None:
        print(f"🏥 ACTUAL SURVIVAL:  {actual_survival:.1f} days")
        error = predicted_days - actual_survival
        print(f"📉 ERROR:            {error:+.1f} days")
    else:
        print("❓ ACTUAL SURVIVAL:  Unknown")
    print("="*50 + "\n")

if __name__ == "__main__":
    # CONFIGURATION
    DATA_PATH = '/kaggle/input/brats20-dataset-training-validation'

    # Run check
    check_patient_survival('BraTS20_Training_003', base_path=DATA_PATH)